# End-to-End Data Pipeline

Full pipeline from raw activity and vehicle logs to the per-interval
counting-process frames the survival models consume. Combines three previously
separate steps:

1. **Cleaning & saving** - merge, segment by user type, filter, and write the
   interim CSVs (`DataCleaner`).
2. **Splitting & imputing** - user-level train/val/test split per segment, with a
   KNN imputer for `vehicle_start_year` fit on train only (`DataSplitter`).
3. **Preparing for modelling** - per-split feature engineering into the interval
   frames (`DataProcessor`).

Each step reads what the previous one wrote, so run the cells top to bottom.


## Setup


In [1]:
# Moving the working directory up to the project root so the src package imports resolve
import os
from pathlib import Path

import pandas as pd
import polars as pl

os.chdir(Path(os.getcwd()).parent)
os.getcwd()

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

In [2]:
import src.constants.paths_to_files_and_folders as paths
from src.constants import paths_to_files_and_folders as const
from src.constants.columns import USER_ID_COL
from src.constants.segments import PERSONAL, PROFESSIONAL
from src.constants.cleaning import (
    DEFAULT_HHI_THRESHOLD,
    DEFAULT_CAR_SHARE_ABS,
    DEFAULT_CAR_SHARE_FRACTION,
)

from src.data_cleaning import DataCleaner
from src.data_splitting import DataSplitter
from src.data_processing import DataProcessor

# 1 · Data Cleaning and Saving

Produces the interim datasets the rest of the project consumes, via
`DataCleaner.get_clean_data`. Two flavours per segment:

- **Personal | Professional RAW** - merged, one-day users removed, end-year imputed, segmented by user
  type. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **All Users RAW** - merged, one-day users removed, end-year imputed. No segmentation. No cutoff or inactivity filtering. Used for the distributional exploration
  in `01`.

- **Personal | Professional FILTERED** - additionally cutoff-filtered, NaN-metadata removed, and truncated
  at the first churn event (adds `churn_adjusted_date`). Used for the interval
  grid and the survival models.


In [3]:
activity_df = pd.read_csv(const.PATH_TO_RAW_ACTIVITY_DATA_1000)
vehicle_df  = pd.read_csv(const.PATH_TO_RAW_VEHICLE_DATA_1000)
data_cleaner = DataCleaner(activity_df, vehicle_df)

# Decided churn threshold (see interval / gap analysis in notebook 01)
CHURN_THRESHOLD_DAYS_PERSONAL = PERSONAL.churn_threshold_days
CHURN_THRESHOLD_DAYS_PROFESSIONAL = PROFESSIONAL.churn_threshold_days

# Shared split criteria
HHI_THRESHOLD       = DEFAULT_HHI_THRESHOLD
CAR_SHARE_ABS       = DEFAULT_CAR_SHARE_ABS
CAR_SHARE_FRACTION  = DEFAULT_CAR_SHARE_FRACTION


## All Users - RAW

In [4]:
merged_all_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=False,
    # return_personal_use_users=True,
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "merged_all_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Number of early churners removed: 198
Rows before early-churner filtering: 560583
Rows after early-churner filtering: 557378
Rows removed: 3205


Cleaning complete
Final rows: 557378
Final unique users: 2557

_Step 4_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\merged_all_users_raw.csv


## Personal - RAW


In [5]:
personal_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by personal users!
Rows after user type filtering: 177113

_Step 4_
Number of early churners removed: 198
Rows before early-churner filtering: 177113
Rows after early-churner filtering: 175747
Rows removed: 1366


Cleaning complete
Final rows: 175747
Final unique users: 2342

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\personal_users_raw.csv


## Professional - RAW


In [6]:
professional_raw = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=True,
    transform_vehicle_end_year_to_present=True,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_raw.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before user type filtering: 560583
Filtering by professional users!
Rows after user type filtering: 64062

_Step 4_
Number of early churners removed: 0
Rows before early-churner filtering: 64062
Rows after early-churner filtering: 64062
Rows removed: 0


Cleaning complete
Final rows: 64062
Final unique users: 215

_Step 5_
Saving File to C:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\Data\interim\professional_users_raw.csv


## Personal - FILTERED

Adds cutoff filtering, NaN-metadata removal, and inactivity truncation on top of
the raw pipeline.


In [7]:
personal_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=True,          # personal
    filter_early_churners=True,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PERSONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before set cutoff date filtering: 560583
Rows after set cutoff date filtering: 559272
Rows removed: 1311

_Step 4_
Rows before user type filtering: 559272
Filtering by personal users!
Rows after user type filtering: 176476

_Step 5_
Rows before vehicle metadata filtering: 176476
Rows after vehicle metadata filtering: 166240
Rows removed: 10236

_Step 6_
Filtering activity after inactivity threshold: 160 days
Rows before inactivity filtering: 166240
Rows after inactivity filtering: 132932
Rows removed: 33308
Unique users before: 2495
Unique users after: 24

## Professional - FILTERED


In [8]:
professional_filtered = data_cleaner.get_clean_data(
    merge_data_frames=True,
    filter_inactivity=True,
    filter_nan_cols=None,
    filter_by_user_type=True,
    return_personal_use_users=False,         # professional
    filter_early_churners=False,
    filter_by_set_cutoff_date=True,
    transform_vehicle_end_year_to_present=True,
    filter_nan_vehicle_metadata=True,
    threshold_value=CHURN_THRESHOLD_DAYS_PROFESSIONAL,
    inverse_hhi_threshold=HHI_THRESHOLD,
    car_share_threshold_abs=CAR_SHARE_ABS,
    car_share_threshold_fraction=CAR_SHARE_FRACTION,
    save_file_to=const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv",
)

_Step 1_
Rows before merging: 567953
Rows after merging: 567953

Removing user_ids that aren't present in vehicle dataset
Rows before user filtering: 567953
Rows after user filtering: 567352

Removing duplicate rows
Rows before deduplication: 567352
Rows after deduplication: 560583

_Step 2_
Transforming missing vehicle_end_year values to current year
Current year used: 2026
Missing vehicle_end_year values filled: 41626

_Step 3_
Rows before set cutoff date filtering: 560583
Rows after set cutoff date filtering: 560468
Rows removed: 115

_Step 4_
Rows before user type filtering: 560468
Filtering by professional users!
Rows after user type filtering: 64062

_Step 5_
Rows before vehicle metadata filtering: 64062
Rows after vehicle metadata filtering: 61148
Rows removed: 2914

_Step 6_
Filtering activity after inactivity threshold: 80 days
Rows before inactivity filtering: 61148
Rows after inactivity filtering: 36477
Rows removed: 24671
Unique users before: 215
Unique users after: 215
Chu

# 2 · Splitting and Imputing Data

Splits each user segment (personal / professional) into train, validation and test
at the user level, then imputes missing `vehicle_start_year` with a KNN imputer fit
on train only. The fitted imputer and encoders are reused on val and test so no
information leaks from the held-out sets into the fill values.


In [9]:
# Filtered interim files produced by the cleaning step, one per user segment
path_to_personal_filtered = const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv"
path_to_professional_filtered = const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv"

In [10]:
path_to_personal_filtered

WindowsPath('C:/Users/Tomas/Desktop/Thesis Stuff/Survival_Analysis_Thesis/Coding/Data/interim/personal_users_filtered.csv')

In [11]:
# One DataSplitter per segment - each holds its own frame for splitting and imputation
data_personal = pl.read_csv(path_to_personal_filtered)
data_splitter_personal = DataSplitter(data_personal)

data_professional = pl.read_csv(path_to_professional_filtered)
data_splitter_professional = DataSplitter(data_professional)

In [12]:
# Personal segment - split, KNN-impute start year, and save the three CSVs
train_df, val_df, test_df = data_splitter_personal.prepare_dataset(data_personal,
                              train_size=0.8,
                              test_size=0.1,
                              val_size=0.1,
                              personal=True,
                              save_path=Path(paths.PATH_TO_INTERIM_DATA))

# Unique user counts per split - confirms the split is at the user level and reproducible
print(train_df[USER_ID_COL].n_unique(),
      val_df[USER_ID_COL].n_unique(),
      test_df[USER_ID_COL].n_unique())

104135 10114 15183
1555 195 195


In [13]:
# Professional segment - same pipeline on the professional splitter and frame
train_df_prof, val_df_prof, test_df_prof = data_splitter_professional.prepare_dataset(data_professional,
                              train_size=0.8,
                              test_size=0.1,
                              val_size=0.1,
                              personal=False,
                              save_path=Path(paths.PATH_TO_INTERIM_DATA))

print(train_df_prof[USER_ID_COL].n_unique(),
      val_df_prof[USER_ID_COL].n_unique(),
      test_df_prof[USER_ID_COL].n_unique())

28072 2362 6043
171 22 22


In [14]:
data_personal["churn_triggered"].sum()

934

# 3 · Preparing Data for Modelling

Reads the imputed train / validation / test splits produced by the splitting step
and runs feature engineering on each, building the per-interval counting-process
frame the survival models consume. Feature engineering runs on each split
independently so no information crosses split boundaries.


In [15]:
# DataProcessor reads the split CSVs itself, so it is constructed without a frame here
processor = DataProcessor(pl.DataFrame())

In [16]:
# Personal segment - feature-engineer each split and save the per-interval frames
train_features, val_features, test_features = processor.prepare_data(
    load_path=const.PATH_TO_INTERIM_DATA,
    personal=True,
    has_validation=True,
    save_path=const.PATH_TO_FINAL_DATA)

print(train_features.shape, val_features.shape, test_features.shape)

c:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\src\data_processing.py:163: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(


(48950, 27) (5756, 27) (6267, 27)


In [17]:
# Professional segment - same feature engineering on the professional splits
train_features_prof, val_features_prof, test_features_prof = processor.prepare_data(
    load_path=const.PATH_TO_INTERIM_DATA,
    personal=False,
    has_validation=True,
    save_path=const.PATH_TO_FINAL_DATA)

print(train_features_prof.shape, val_features_prof.shape, test_features_prof.shape)

(4706, 27) (491, 27) (738, 27)
